# **Done on Collab**



In [4]:
!pip install transformers
!pip install "transformers[torch]"


In [5]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [6]:
train_data = pd.read_csv("/content/samsum-train.csv")
val_data = pd.read_csv("/content/samsum-validation.csv")

In [7]:
train_data

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [8]:
train_data.shape

(14732, 3)

In [9]:
val_data.shape

(818, 3)

In [10]:
train_data = train_data.sample(n=4000 , random_state =42).reset_index(drop=True)
val_data = val_data.sample(n=400 , random_state =42).reset_index(drop=True)


Data Preprocessing

In [11]:
import re # regular expressions

def clean_data(text):
  text = re.sub(r"'\r\n'", " ", text)
  text = re.sub(r"\s+", " ", text)
  text = re.sub(r"<.*?>", " ", text)
  text = text.strip().lower() # Added parentheses here
  return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


Tokenization - txt to No.

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
def tokenize(data):
  inputs = tokenizer(data["dialogue"] , padding="max_length" , truncation=True , max_length=512)
  targets = tokenizer(data["summary"] , padding="max_length" , truncation=True , max_length=150)



  inputs["labels"] = targets["input_ids"]
  return inputs

In [15]:
train_dataset = train_data.apply(tokenize , axis = 1).tolist()
val_dataset = val_data.apply(tokenize , axis = 1).tolist()

In [16]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
# len(train_dataset[0]["input_ids"])
type(train_dataset)
type(val_dataset)

list

Model

In [18]:
model  = T5ForConditionalGeneration.from_pretrained("t5-small") # fenerate acc to our input

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")


print("device: " , device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

Arguments for transformers

In [20]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 1,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500 #  learning increases
    )

In [21]:
# using transformer trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.577381,0.374473


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=3.577380615234375, metrics={'train_runtime': 204.0988, 'train_samples_per_second': 19.598, 'train_steps_per_second': 2.45, 'total_flos': 541367205888000.0, 'train_loss': 3.577380615234375, 'epoch': 1.0})

In [26]:
model.save_pretrained("./saved_text.summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [27]:
tokenizer.save_pretrained("./saved_text.summary_model")

('./saved_text.summary_model/tokenizer_config.json',
 './saved_text.summary_model/tokenizer.json')

In [33]:
model = T5ForConditionalGeneration.from_pretrained("./saved_text.summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_text.summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Testing

In [38]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) # clean
  inputs = tokenizer(dialogue , padding = "max_length" , truncation=True , max_length=512 , return_tensors="pt").to(device) # tokenize

  model.to(device)
  # Move inputs to the same device as the model
  input_ids = inputs["input_ids"].to(device)
  attention_mask = inputs["attention_mask"].to(device)

  targets = model.generate(
      input_ids = input_ids,
      attention_mask = attention_mask,
      max_length = 150,
      num_beams = 4, #compare 4 output summary to plosh summary and give one
      early_stopping = True
  )

  # Fixed the syntax error here by passing skip_special_tokens as a proper argument
  summary = tokenizer.decode(targets[0], skip_special_tokens=False) # r normal text # try with tokens = true.
  return summary

In [34]:
# Test the model on a sample from the validation set
sample_dialogue = val_data['dialogue'][0]
print("Original Dialogue:\n", sample_dialogue)

summary = summarize_dialogue(sample_dialogue)
print("\nGenerated Summary:\n", summary)

Original Dialogue:
 edd: wow, did you hear that they're transferring us to a different department? rose: whaaaaat :o rose: no! where'd you hear that? edd: well, it's quite official edd: anderson just told us rose: and do you know what it changes for us? edd: they won't change the professors edd: but i know the paperwork will get trickier rose: and i guess that is a move that is supposed to make everything easier edd: yeah, guess so edd: they have a funny way of understanding 'to make things easier'

Generated Summary:
 edd and anderson are transferring the professors to a different department. edd and edd have a funny way of understanding to make things easier.


In [39]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  <pad> experts are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. ai systems are becoming more capable due to advances in deep learning and access to large datasets.</s>
